In [1]:
import sys
from pathlib import Path

HERE = Path().resolve().parent  # notebooks/ → repo root

sys.path.insert(0, str(HERE / "src"))
sys.path.insert(0, str((HERE.parents[1] / "Wound ontology" / "src").resolve()))

from wound_ontology.tbox_abox_reasoner.for_owl.tbox import TBox
from wound_ontology.tbox_abox_reasoner.for_rdflib.abox import ABox

TBOX_FILE = HERE / "ontologies" / "ofl_2.1.0.owl"
OUTPUT    = HERE / "ontologies" / "ofl_abox.owl"
PATIENTS  = 1
SEED      = 42
ID_START  = 20_000

tbox = TBox().load(str(TBOX_FILE))
gen  = ABox(tbox, id_start=ID_START)
gen.generate(patients_per_combination=PATIENTS, seed=SEED)
gen.save(str(OUTPUT))
print(f"Saved → {OUTPUT}  ({OUTPUT.stat().st_size:,} bytes)")


[TBox] owlready2 warning (OwlReadyOntologyParsingError) — partial load
[TBox] Loaded: /Users/Strubbelig/Library/Mobile Documents/com~apple~CloudDocs/Forschung/Flap ontology/VS Code output/ofl_2.1.0.owl


[ABox] TBox: 191 rich flaps, 47 with origin, 58 pedicles, 206 total origin-leaf flap classes


[ABox] 3,910 flap instances, 51,980 individuals, 243,657 triples


Saved → /Users/Strubbelig/Library/Mobile Documents/com~apple~CloudDocs/Forschung/Flap ontology/VS Code output/ofl_abox.owl  (28,046,113 bytes)


In [ ]:
from wound_ontology.tbox_abox_reasoner.reasoner.reasoner import Reasoner

ABOX_INFERRED = HERE / "ontologies" / "ofl_abox_inferred.owl"

consistent = Reasoner().expand_owlrl(
    tbox_file   = str(TBOX_FILE),
    abox_file   = str(OUTPUT),
    output_file = str(ABOX_INFERRED),
)
print(f"Consistent: {consistent}")


In [ ]:
from pyshacl import validate

SHACL_FILE = HERE / "shacl" / "ofl_local_flap.shacl.ttl"

conforms, report_graph, report_text = validate(
    data_graph        = str(ABOX_INFERRED),
    shacl_graph       = str(SHACL_FILE),
    data_graph_format = "xml",
    shacl_graph_format= "turtle",
    inference         = "none",   # already materialised by OWL RL
    serialize_report_graph = "turtle",
)

print(f"Conforms: {conforms}")
print(report_text)
